In [1]:
library(MLmetrics)
library(randomForest)
set.seed(2) 

Warning message:
"le package 'MLmetrics' a été compilé avec la version R 4.2.3"

Attachement du package : 'MLmetrics'


L'objet suivant est masqué depuis 'package:base':

    Recall


Warning message:
"le package 'randomForest' a été compilé avec la version R 4.2.3"
randomForest 4.7-1.1

Type rfNews() to see new features/changes/bug fixes.



In [2]:
ConfusionMatrix <- function(y_pred, y_true) {
  Confusion_Mat <- table(y_true, y_pred)
  return(Confusion_Mat)
}
 
ConfusionDF <- function(y_pred, y_true) {
  Confusion_DF <- transform(as.data.frame(ConfusionMatrix(y_pred, y_true)),
                            y_true = as.character(y_true),
                            y_pred = as.character(y_pred),
                            Freq = as.integer(Freq))
  return(Confusion_DF)
}
 
Precision_micro <- function(y_true, y_pred, labels = NULL) {
  Confusion_DF <- ConfusionDF(y_pred, y_true)
 
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred))
  # this is not bulletproof since there might be labels missing (in strange cases)
  # in strange cases where they existed in training set but are missing from test ground truth and predictions.
 
  TP <- c()
  FP <- c()
  for (i in c(1:length(labels))) {
    positive <- labels[i]
   
    # it may happen that a label is never predicted (missing from y_pred) but exists in y_true
    # in this case ConfusionDF will not have these lines and thus the simplified code crashes
    # TP[i] <- as.integer(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"])
    # FP[i] <- as.integer(sum(Confusion_DF[which(Confusion_DF$y_true!=positive & Confusion_DF$y_pred==positive), "Freq"]))
   
    # workaround:
    # i don't want to change ConfusionDF since i don't know if the current behaviour is a feature or a bug.
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"]
    TP[i] <- if (length(tmp)==0) 0 else as.integer(tmp)
   
    tmp <- Confusion_DF[which(Confusion_DF$y_true!=positive & Confusion_DF$y_pred==positive), "Freq"]
    FP[i] <- if (length(tmp)==0) 0 else as.integer(sum(tmp))
  }
  Precision_micro <- sum(TP) / (sum(TP) + sum(FP))
  return(Precision_micro)
}
 
Recall_micro <- function(y_true, y_pred, labels = NULL) {
  Confusion_DF <- ConfusionDF(y_pred, y_true)
 
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred))
  # this is not bulletproof since there might be labels missing (in strange cases)
  # in strange cases where they existed in training set but are missing from test ground truth and predictions.
 
  TP <- c()
  FN <- c()
  for (i in c(1:length(labels))) {
    positive <- labels[i]
   
    # short version, comment out due to bug or feature of Confusion_DF
    # TP[i] <- as.integer(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"])
    # FP[i] <- as.integer(sum(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred!=positive), "Freq"]))
   
    # workaround:
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"]
    TP[i] <- if (length(tmp)==0) 0 else as.integer(tmp)
 
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred!=positive), "Freq"]
    FN[i] <- if (length(tmp)==0) 0 else as.integer(sum(tmp))
  }
  Recall_micro <- sum(TP) / (sum(TP) + sum(FN))
  return(Recall_micro)
}
 
F1_Score_micro <- function(y_true, y_pred, labels = NULL) {
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred)) # possible problems if labels are missing from y_*
  Precision <- Precision_micro(y_true, y_pred, labels)
  Recall <- Recall_micro(y_true, y_pred, labels)
  F1_Score_micro <- 2 * (Precision * Recall) / (Precision + Recall)
  return(F1_Score_micro)
}

In [3]:
data<-read.csv("train_values.csv",stringsAsFactors = T)
data_labels<-read.csv("train_labels.csv",stringsAsFactors = T)
datam<-merge(data,data_labels,by=c('building_id','building_id'))

We perform here a k-fold CV on randomForest for the full dataset with the optimal parameters obtained from randomForestTune.ipynb

NOTE TO SELF: ADD PROGRESSBAR

In [5]:
k = 10
n_trees <- 1000
features_per_split <- 12

target_variable <- ncol(datam)-1
id_variable <- match('building_id', colnames(datam))

accuracy_vec <- array(0,k)
threshold <- 0.5

# 1. Shuffle the dataset randomly.
datam_idx <- sample(1:nrow(datam))

# 2. Split the dataset into k groups
max <- ceiling(nrow(datam)/k)
splits <- split(datam_idx, ceiling(seq_along(datam_idx)/max))

pb <- txtProgressBar(min = 0, max = k, style = 3)
# 3. For each unique group:
for (i in 1:k){

    #3.1 Take the group as a hold out or test data set
    test_data <- datam[splits[[i]],-id_variable]


    #3.2 Take the remaining groups as a training data set
    train_data <- datam[-splits[[i]],-id_variable]   

    model <- randomForest(x=train_data[,-c(target_variable)],
                        y=as.factor(train_data[,c(target_variable)]),
                        ntree=n_trees,mtry=features_per_split,keep.forest=TRUE,importance=TRUE)
    yhat<-predict(model,test_data[,-c(target_variable)])                      
    accuracy_vec[i]<-F1_Score_micro(as.factor(test_data[,c(target_variable)]),yhat)
    setTxtProgressBar(pb, i)
    print(paste("F1-Score Micro -",i,"fold:",accuracy_vec[i]))
    rm(c('model'))
}

#4. Summarize the accuracy of the model using the sample of model evaluation scores
print(paste("Mean F1-Score Micro:",mean(accuracy_vec)))

  |=======                                                               |  10%[1] "F1-Score Micro - 1 fold: 0.73819116687771"


ERROR: Error: impossible d'allouer un vecteur de taille 3.5 Go
